# SIMPLIcity: EfficientNet-B0 Expert Enhancement Runner (V3 Final vs V4 Expert)
### Targeting >= 0.9700 Macro F1 with GeM Pooling, Frozen BatchNorm, Multi-Class Focal Loss, and Anisotropic Resizing

This notebook trains and evaluates the optimized **EfficientNet-B0** pipeline on the SIMPLIcity dataset (1,000 images, 10 classes) using **Dual NVIDIA Tesla T4 GPUs** (`torch.nn.DataParallel`).

---
### Evolution & Strategic Levers:
1. **V1 Pretrained Baseline**: Accuracy 96.50% | Macro F1 0.9652 | 7 errors.
2. **V2 Letterbox (240x240)**: Accuracy 95.50% | Macro F1 0.9539 | 9 errors (dead padding degraded GAP).
3. **V3 Streamlined Final**: Accuracy 96.00% | Macro F1 0.9605 | 8 errors (canonical 224x224, linear head, label smoothing 0.04).
4. **V4 Expert Architecture (Target >= 0.9700)**:
   * **Tuas 1 (Anisotropic Geometry)**: Direct `Resize((224, 224))` without 56px CenterCrop to preserve mountain summits and building spires.
   * **Tuas 2 (Frozen BatchNorm)**: Keeps ImageNet running statistics in `eval()` mode to prevent small-batch ($N=32$) noise.
   * **Tuas 3 (Generalized Mean Pooling - GeM)**: Replaces GAP with `GeMPool2d(p=3.0)` to focus on salient structural edges over diffuse blue sky.
   * **Tuas 4 (Multi-Class Focal Loss)**: Sets $\gamma = 1.5$ to suppress gradients on the 6 perfect classes and focus 100% on the 4 boundary classes.
   * **Tuas 5 (3-Tier Differential LR)**: Differential fine-tuning ($3.0 \times 10^{-5}$ backbone vs $3.0 \times 10^{-4}$ head).


## 1. Verify GPU Acceleration & Dual T4 Topology
Checks CUDA availability and device properties across both Tesla T4 GPUs.

In [ ]:
!nvidia-smi

import os

import torch

print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
count = torch.cuda.device_count()
print(f"GPU Count       : {count}")
for i in range(count):
    name = torch.cuda.get_device_name(i)
    props = torch.cuda.get_device_properties(i)
    mem = getattr(props, "total_memory", getattr(props, "total_mem", 0)) / (1024**3)
    print(f"  GPU {i}: {name} ({mem:.1f} GB VRAM)")
if count > 1:
    print(">> Dual GPU detected. DataParallel acceleration enabled across all cards <<")

## 2. Setup Environment & Sync Repository
Pulls the latest code updates from the GitHub repository.

In [ ]:
import sys
from pathlib import Path

# 1. Ensure working directory is project root
if os.path.exists("src/simplicity_cnn"):
    repo_root = Path.cwd()
elif os.path.exists("../src/simplicity_cnn"):
    %cd ..
    repo_root = Path.cwd()
elif os.path.exists("cnn-SIMPLIcity/src/simplicity_cnn"):
    %cd cnn-SIMPLIcity
    repo_root = Path.cwd()
elif Path("/kaggle/working/cnn-SIMPLIcity/src/simplicity_cnn").exists():
    %cd /kaggle/working/cnn-SIMPLIcity
    repo_root = Path.cwd()
else:
    print("Cloning latest repository from GitHub...")
    !git clone -b main https://github.com/Arfiadi/cnn-SIMPLIcity.git
    %cd cnn-SIMPLIcity
    repo_root = Path.cwd()

# 2. Pull latest updates from main
!git pull origin main

# 3. Configure sys.path, PYTHONPATH, and install package in editable mode
src_path = str((repo_root / "src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)
os.environ["PYTHONPATH"] = f"{src_path}:{os.environ.get('PYTHONPATH', '')}"

%pip install -q -e . --no-deps
print(f"Active working directory: {repo_root}")
print(f"Configured sys.path and PYTHONPATH with: {src_path}")

## 3. Dataset Verification
Auto-detects and audits dataset split disjointness and file counts.

In [ ]:
!python scripts/audit_data.py

## 4. Inspect Configuration (V4 Expert vs V3 Final)
Displays the calibrated parameters for V4 Expert targeting $\ge 0.9700$ Macro F1.


In [ ]:
from simplicity_cnn.config.schema import print_config_summary

print(">> V4 EXPERT CONFIGURATION (Target >= 0.9700) <<")
cfg_v4 = print_config_summary("configs/efficientnet_b0_v4_expert.yaml")

print("\n>> V3 FINAL CONFIGURATION (Baseline Comparative) <<")
cfg_v3 = print_config_summary("configs/efficientnet_b0.yaml")

## 5. Execute Training: EfficientNet-B0 (V4 Expert)
Trains EfficientNet-B0 with GeM Pooling ($p=3.0$), Frozen BatchNorm, Multi-Class Focal Loss ($\gamma=1.5$), and Differential LR across Dual Tesla T4 GPUs.

> *To train V3 Final instead, replace the config argument with `configs/efficientnet_b0.yaml`.*


In [ ]:
# Execute V4 Expert Training
!python scripts/train.py --config configs/efficientnet_b0_v4_expert.yaml --multi-gpu --backbone-lr-ratio 0.1

## 6. Training Curves & Convergence Diagnostics
Plots loss convergence, accuracy progression, and confusion matrix from the latest training run.


In [ ]:
from pathlib import Path

from IPython.display import Image, display

# Check V4 Expert output first, fallback to V3 Final
v4_dir = Path("outputs/efficientnet_b0_v4_expert")
v3_dir = Path("outputs/efficientnet_b0_v3_final")
out_dir = (
    v4_dir if v4_dir.exists() and (v4_dir / "gradient_descent_summary.png").exists() else v3_dir
)

print(f"Displaying diagnostic curves from: {out_dir}")
summary_plot = out_dir / "gradient_descent_summary.png"
cm_plot = out_dir / "evaluation" / "confusion_matrix.png"

if summary_plot.exists():
    print("=== Training Summary Curves ===")
    display(Image(filename=str(summary_plot)))

if cm_plot.exists():
    print("=== Test Set Confusion Matrix ===")
    display(Image(filename=str(cm_plot)))

## 7. Multi-Way Ablation Analysis (Baseline vs V2 vs V3 vs V4 Expert)
Compares per-class F1 scores and overall accuracy across all iterations to verify the $\ge 0.9700$ target.


In [ ]:
import json
from pathlib import Path

import pandas as pd

v4_report = Path("outputs/efficientnet_b0_v4_expert/evaluation/classification_report.json")
v3_report = Path("outputs/efficientnet_b0_v3_final/evaluation/classification_report.json")

report_file = v4_report if v4_report.exists() else v3_report
run_label = "V4 (Expert)" if v4_report.exists() else "V3 (Final)"

if report_file.exists():
    with open(report_file, "r", encoding="utf-8") as f:
        report = json.load(f)

    classes = [
        "People",
        "Beach",
        "Building",
        "Bus",
        "Dinosaur",
        "Elephant",
        "Flower",
        "Horse",
        "Mountain",
        "Food",
    ]
    baseline_f1 = {
        "People": 0.9744,
        "Beach": 0.8889,
        "Building": 0.8889,
        "Bus": 1.0000,
        "Dinosaur": 1.0000,
        "Elephant": 1.0000,
        "Flower": 1.0000,
        "Horse": 1.0000,
        "Mountain": 0.9000,
        "Food": 1.0000,
    }
    v2_f1 = {
        "People": 0.9756,
        "Beach": 0.8837,
        "Building": 0.8571,
        "Bus": 0.9756,
        "Dinosaur": 1.0000,
        "Elephant": 0.9756,
        "Flower": 1.0000,
        "Horse": 1.0000,
        "Mountain": 0.8718,
        "Food": 1.0000,
    }
    v3_f1 = {
        "People": 0.9744,
        "Beach": 0.8696,
        "Building": 0.8889,
        "Bus": 1.0000,
        "Dinosaur": 1.0000,
        "Elephant": 1.0000,
        "Flower": 1.0000,
        "Horse": 1.0000,
        "Mountain": 0.8718,
        "Food": 1.0000,
    }
    rows = []
    for c in classes:
        if c in report:
            prec = report[c]["precision"]
            rec = report[c]["recall"]
            f1_curr = report[c]["f1-score"]
            f1_base = baseline_f1.get(c, 0.0)
            f1_v3_val = v3_f1.get(c, 0.0)
            delta_base = (f1_curr - f1_base) * 100
            delta_v3 = (f1_curr - f1_v3_val) * 100
            rows.append(
                {
                    "Class": c,
                    "Precision": f"{prec:.4f}",
                    "Recall": f"{rec:.4f}",
                    "Baseline F1": f"{f1_base:.4f}",
                    "V3 F1": f"{f1_v3_val:.4f}",
                    run_label: f"{f1_curr:.4f}",
                    "Delta vs Base": f"{delta_base:+.2f}%",
                    "Delta vs V3": f"{delta_v3:+.2f}%",
                }
            )
    df = pd.DataFrame(rows)
    print("=" * 100)
    print(f" EFFICIENTNET-B0: COMPREHENSIVE ABLATION STUDY (BASELINE vs V2 vs V3 vs {run_label})")
    print("=" * 100)
    print(df.to_string(index=False))
    print("-" * 100)
    acc = report.get("accuracy", 0) * 100
    macro_f1 = report.get("macro avg", {}).get("f1-score", 0)
    print(" [V1 Baseline]   Accuracy: 96.50% | Macro F1: 0.9652 | Errors: 7/200")
    print(" [V2 Letterbox]  Accuracy: 95.50% | Macro F1: 0.9539 | Errors: 9/200")
    print(" [V3 Final]      Accuracy: 96.00% | Macro F1: 0.9605 | Errors: 8/200")
    print(f" [{run_label:<12}] Accuracy: {acc:.2f}% | Macro F1: {macro_f1:.4f}")
    print("=" * 100)
else:
    print(f"Evaluation report not found at {report_file}. Run training first.")

## 8. Misclassification Error Inspection
Inspects remaining test errors with confidence scores against prior iterations.


In [ ]:
import json
from pathlib import Path

v4_err = Path("outputs/efficientnet_b0_v4_expert/evaluation/misclassified_samples.json")
v3_err = Path("outputs/efficientnet_b0_v3_final/evaluation/misclassified_samples.json")
err_file = v4_err if v4_err.exists() else v3_err
active_label = "V4 Expert" if v4_err.exists() else "V3 Final"

if err_file.exists():
    with open(err_file, "r", encoding="utf-8") as f:
        errors = json.load(f)
    print(f"=== Misclassified Samples: {active_label} ({len(errors)} / 200 errors) ===")
    print("Benchmark: Baseline (7 errors) | V2 (9 errors) | V3 (8 errors) | Target <= 5 errors")
    for i, err in enumerate(errors, 1):
        fn = err["filename"]
        t_lbl = err["true_label"]
        p_lbl = err["predicted_label"]
        c_val = err["confidence"]
        print(
            f"  [{i:02d}] File: {fn:<8} | True: {t_lbl:<10} -> Predicted: {p_lbl:<10} (Conf: {c_val:.4f})"
        )
else:
    print(f"Error report not found at {err_file}.")